# Day 5: All sources together

The COO asked you for a quick company pulse before the all-hands meeting. They need inventory risk from Fabric, a summary of active email comms about the stock situation, the escalation protocol from company policy, and a brief health benefits highlight for the employee segment. All in one answer.

**🎯 Mission**
- Build a **4-source knowledge base** combining HR docs, health docs, Fabric IQ, and Work IQ
- Fire a single query that simultaneously spans structured product data, personal workplace comms, policy documents, and health benefits
- Inspect the activity log to see how the agent decomposes one question into parallel sub-queries across all four sources


## The building blocks

This is the capstone notebook. Everything you built in Parts 1–4 comes together here.

- **4-source retrieval**: The agent routes sub-queries to any combination of HR docs, health docs, Fabric IQ, or Work IQ in a single request.

## Step 1: Sign in with Azure CLI

This notebook uses `AzureCliCredential` to fetch a delegated query token for both Work IQ and Fabric IQ. Sign in before running any other cells.

Run this in a terminal:

```shell
az login --tenant <YOUR_TENANT_ID>
```

To find your tenant ID:
- Check the `AZURE_TENANT_ID` value already in your `.env` file
- Or open the [Azure Portal](https://portal.azure.com), search for **Microsoft Entra ID**, and find it on the Overview page under **Basic information**
- Or run `az account show --query tenantId -o tsv` if you are already signed in to the correct account

## Step 2: Set up environment variables

Load the configuration for your Azure AI Search, Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
user_credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID, process_timeout=60)

print("Environment variables loaded")

## Step 3: Acquire your delegated user token

In order for the knowledge base to make queries to a Work IQ knowledge source, it requires a delegated user token.

In this step, you will acquire a delegated user token for the Azure AI Search scope, so that the knowledge base is allowed to access Work IQ on your behalf during retrieval later.

This code uses `AzureCliCredential`, so the token comes from the Azure CLI account you signed in with before starting the notebook. 

In [ ]:
import base64

token_obj = user_credential.get_token("https://search.azure.com/.default")
user_token = token_obj.token

# Decode JWT payload to verify the signed-in account
payload_b64 = user_token.split(".")[1]
payload_b64 += "=" * (-len(payload_b64) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload_b64))

unique_name = claims.get("unique_name")
if unique_name:
    print("Acquired token for Azure AI Search query source")
    print(f"Signed in as: {unique_name}")
else:
    print("❌ No unique_name found in the token. The CLI may be signed in to the wrong account.")
    print(f"   Run: az login --tenant {AZURE_TENANT_ID}")
    print("   Then re-run this cell.")


## Step 4: Create all knowledge sources

For this knowledge base, you'll create four knowledge sources: the same two search-index sources used earlier, plus Work IQ and Fabric Ontology sources.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `workiq-knowledge-source`: Points to Work IQ for workplace context
* `fabric-ontology-knowledge-source`: Points to the Fabric workspace and ontology used for structured operational data

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology knowledge source",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")

## Step 5: Create the combined knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-work-fabric-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Zava KB combining restored indexes, Work IQ, and Fabric Ontology",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context, Fabric Ontology for structured operational data, and restored indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## Step 6: Query the knowledge base

Ask a four-part all-hands question that spans all four sources at once:

* _"Which product categories have the most critically low stock levels?"_ → **Fabric IQ**
* _"Do I have recent emails about the claw hammer stock situation?"_ → **Work IQ**
* _"How do I escalate urgent operational issues to the executive team?"_ → `hrdocs`
* _"What mental health coverage does Zava offer employees?"_ → `healthdocs`

> **Note:** Combined Work IQ and Fabric IQ requests may return `206 Partial Content` when one source succeeds and another is partially unavailable. That is still a useful result. Check the activity payload for details.

> 💡 **Try your own:** Edit the `question` variable below and re-run the cell.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = (
    "For the Friday all-hands, I need four things: "
    "1) Using the Zava DIY Fabric ontology, which product categories have the most critically low stock levels? "
    "2) Do I have any recent work emails about the Professional Claw Hammer stock situation or other inventory issues? "
    "3) What does the Zava employee handbook say about how to escalate urgent operational issues to the executive team? "
    "4) What mental health coverage does Zava offer employees? I want to mention our benefits briefly at the all-hands."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=300,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)

# Extract synthesized answer from response messages

for message in result.response or []:
    for content in message.content or []:
        text = getattr(content, "text", None)
        if text:
            display(Markdown(text))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base so you can inspect which sources contributed to the answer.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

### 🔍 Source Hunt

Look at the references printed above. Can you identify all four source types?

1. Which references came from **Fabric IQ** (structured product entities)?
2. Which references came from **Work IQ** (your M365 emails/Teams)?
3. Which references came from the **HR index** (policy documents)?
4. Which references came from **healthdocs** (health benefits documents)?

> **Grand challenge:** Count how many references came from each source. Which source contributed the most to the final answer?

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

## ✅ Mission complete

**What you built across the lab:**

* ✓ **4-source Knowledge Base**: A single KB spanning HR docs, health docs, structured Fabric product data, and live M365 workplace communications.
* ✓ **Parallel multi-source retrieval**: One question decomposed into sub-queries routed simultaneously to all four source types.
* ✓ **Full agentic RAG pipeline**: From pre-built indexes to Fabric ontology to personal email, the KB cited and synthesized every source in one response.